<a href="https://colab.research.google.com/github/khalid-saqr/picoNewton/blob/main/picoNewton_v4/notebooks/picoNewton_v4_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# picoNewton_v4 — full parametric production study

This is the single Google Colab entry point for the six-artery anisotropic Womersley → membrane–cortex → Piezo1 study. Run all cells once from top to bottom in a standard Colab CPU runtime. The notebook creates a unique UTC-stamped Google Drive directory, installs the reviewed package from GitHub, runs the package tests, solves the full-resolution model, executes the declared parameter ensemble, produces Nature-compatible multi-panel figures, and archives all data, provenance, environment metadata, and checksums.

## Scientific question

Does anisotropic near-wall forcing provide mechanosensory information that is distinguishable from wall shear stress across the aortic root, thoracic aorta, femoral, carotid, iliac, and brachial arteries, and are current-based conclusions robust to the declared uncertain membrane and endpoint parameters?

## Vector-resolved membrane and endpoint model

Wall shear stress, signed transverse Lamb force, and nonnegative force exposure remain separate physical inputs. Normal and tangential mechanics are transferred through separate passive viscoelastic branches to apical and junctional membrane domains. Piezo1 current is the primary decision endpoint. The calcium-scale output is exploratory and cannot rescue a failed current decision.

## Full numerical resolution

The production workflow is fixed at radial order 150, 2,048 time points per cardiac cycle, 256 near-wall integration nodes, six arteries, and the complete direct, matched-load, surrogate, anisotropy, harmonic, and elastic control matrix.

## Parametric design

The full hydrodynamics are computed once and archived. A deterministic 13-scenario one-at-a-time ensemble then reuses those exact 2,048-point forcing waveforms and re-solves the membrane and Piezo1 dynamics while varying localization area, force-transfer fraction, channel count, apical channel fraction, fast viscoelastic fraction, and pressure ceiling. The design is declared before execution and is not retuned after results are observed.

## Nature-compatible figures

Each complete multi-panel figure is generated at 183 mm double-column width and below 170 mm height. Text is a standard sans-serif family at 5–7 pt, panel labels are lowercase bold 8 pt, line widths are at least 1 pt, backgrounds are white, and panels are arranged alphabetically. Figures are exported as editable PDF and SVG and as 600 dpi PNG and LZW-compressed TIFF.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Run configuration

`REPO_REF="main"` ensures the notebook executes the merged production script. The resolved commit SHA is recorded in every run, so the exact source remains reproducible.

In [ ]:
REPO_URL = "https://github.com/khalid-saqr/picoNewton.git"
REPO_REF = "main"
PACKAGE_SUBDIR = "picoNewton_v4"
NUMERICAL_PROFILE = "full"
RUN_PACKAGE_TESTS = True
EXPECTED_TEST_COUNT = 18

## 3. Create immutable runtime directories

Computation uses local Colab storage for speed. A unique persistent directory is created immediately under `MyDrive/picoNewton_v4_runtime/runs/`, and command logs are mirrored there as they are produced.

In [ ]:
from __future__ import annotations

import hashlib
import json
import secrets
import shutil
import subprocess
import sys
import traceback
from datetime import datetime, timezone
from pathlib import Path

RUN_ID = f"{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_UTC')}_{secrets.token_hex(2)}"
DRIVE_PARENT = Path("/content/drive/MyDrive/picoNewton_v4_runtime/runs")
DRIVE_RUN_ROOT = DRIVE_PARENT / RUN_ID
LOCAL_ROOT = Path("/content/picoNewton_v4_work") / RUN_ID
REPOSITORY_DIR = LOCAL_ROOT / "repository"
PACKAGE_DIR = REPOSITORY_DIR / PACKAGE_SUBDIR
LOCAL_OUTPUT = LOCAL_ROOT / "production_output"
LOCAL_LOGS = LOCAL_ROOT / "logs"

DRIVE_PARENT.mkdir(parents=True, exist_ok=True)
DRIVE_RUN_ROOT.mkdir(parents=False, exist_ok=False)
(DRIVE_RUN_ROOT / "logs").mkdir(parents=True, exist_ok=True)
LOCAL_LOGS.mkdir(parents=True, exist_ok=True)


def run_logged(command, *, cwd=None, log_name):
    command = [str(item) for item in command]
    print("\n$", " ".join(command), flush=True)
    lines = []
    with subprocess.Popen(
        command,
        cwd=cwd,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
    ) as process:
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="")
            lines.append(line)
        return_code = process.wait()
    text = "".join(lines)
    (LOCAL_LOGS / log_name).write_text(text, encoding="utf-8")
    (DRIVE_RUN_ROOT / "logs" / log_name).write_text(text, encoding="utf-8")
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)
    return text


def record_failure(stage, error):
    payload = {
        "stage": stage,
        "run_id": RUN_ID,
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "error_type": type(error).__name__,
        "error": str(error),
        "traceback": traceback.format_exc(),
    }
    (DRIVE_RUN_ROOT / "logs" / "failure.json").write_text(
        json.dumps(payload, indent=2, sort_keys=True) + "\n", encoding="utf-8"
    )

print("Run ID:", RUN_ID)
print("Persistent output:", DRIVE_RUN_ROOT)

## 4. Clone and install the merged package

The repository is cloned from GitHub, the selected ref is resolved, and the exact commit is recorded. The editable install ensures that the notebook and production script use the source in the cloned repository.

In [ ]:
try:
    run_logged(["git", "clone", "--branch", REPO_REF, "--single-branch", REPO_URL, REPOSITORY_DIR], log_name="git_clone.log")
    RESOLVED_COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPOSITORY_DIR, text=True).strip()
    run_logged([sys.executable, "-m", "pip", "install", "-e", str(PACKAGE_DIR) + "[dev]"], log_name="pip_install.log")

    import piconewton_v4
    from piconewton_v4.workflow import run_workflow
    from piconewton_v4.scientific_study import run_scientific_study

    MODULE_PATH = Path(piconewton_v4.__file__).resolve()
    EXPECTED_SOURCE = (PACKAGE_DIR / "src" / "piconewton_v4").resolve()
    if EXPECTED_SOURCE not in MODULE_PATH.parents:
        raise RuntimeError(f"Package imported from {MODULE_PATH}, expected under {EXPECTED_SOURCE}")
    if not callable(run_workflow) or not callable(run_scientific_study):
        raise RuntimeError("Required workflow entry points are unavailable")
    print("Resolved commit:", RESOLVED_COMMIT)
    print("Installed module:", MODULE_PATH)
except Exception as error:
    record_failure("install", error)
    raise

## 5. Run the package test suite

The model is not executed unless the repository tests pass. The complete pytest output is stored in Drive.

In [ ]:
try:
    if RUN_PACKAGE_TESTS:
        output = run_logged([sys.executable, "-m", "pytest", "-q", "-ra"], cwd=PACKAGE_DIR, log_name="pytest.log")
        if f"{EXPECTED_TEST_COUNT} passed" not in output:
            raise RuntimeError(f"Expected {EXPECTED_TEST_COUNT} passing tests; inspect pytest.log")
        print("All package tests passed.")
except Exception as error:
    record_failure("tests", error)
    raise

## 6. Execute the full parametric production workflow

The repository script runs the full current-primary scientific study, the 13-scenario full-resolution membrane–Piezo1 ensemble, three Nature-compatible multi-panel figures, figure legends, provenance, environment capture, and SHA-256 checksums.

In [ ]:
try:
    PRODUCTION_SCRIPT = PACKAGE_DIR / "scripts" / "run_colab_production.py"
    if not PRODUCTION_SCRIPT.is_file():
        raise FileNotFoundError(PRODUCTION_SCRIPT)
    run_logged([
        sys.executable,
        PRODUCTION_SCRIPT,
        "--package-root", PACKAGE_DIR,
        "--output", LOCAL_OUTPUT,
        "--profile", NUMERICAL_PROFILE,
        "--repository-commit", RESOLVED_COMMIT,
    ], cwd=PACKAGE_DIR, log_name="production.log")
except Exception as error:
    record_failure("production", error)
    raise

## 7. Validate the completed local run

In [ ]:
required = (
    LOCAL_OUTPUT / "FINAL_MANIFEST.json",
    LOCAL_OUTPUT / "SHA256SUMS.json",
    LOCAL_OUTPUT / "scientific_study" / "model_outputs" / "six_artery_summary.csv",
    LOCAL_OUTPUT / "scientific_study" / "assessment" / "primary_current_decisions.csv",
    LOCAL_OUTPUT / "scientific_study" / "assessment" / "scientific_assessment.json",
    LOCAL_OUTPUT / "scientific_study" / "hydrodynamics" / "physical_forcing_waveforms.npz",
    LOCAL_OUTPUT / "parameter_study" / "parameter_scenarios.csv",
    LOCAL_OUTPUT / "parameter_study" / "parametric_artery_results.csv",
    LOCAL_OUTPUT / "parameter_study" / "parameter_scenario_summary.csv",
    LOCAL_OUTPUT / "parameter_study" / "parametric_current_waveforms.npz",
    LOCAL_OUTPUT / "figures" / "Figure_1_forcing_and_baseline_response.pdf",
    LOCAL_OUTPUT / "figures" / "Figure_2_current_primary_assessment.pdf",
    LOCAL_OUTPUT / "figures" / "Figure_3_parametric_robustness.pdf",
)
missing = [path for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError("Missing production outputs:\n" + "\n".join(str(path) for path in missing))
manifest = json.loads((LOCAL_OUTPUT / "FINAL_MANIFEST.json").read_text())
if manifest.get("status") != "completed_with_claims_disabled":
    raise RuntimeError(json.dumps(manifest, indent=2, sort_keys=True))
if manifest.get("package_commit") != RESOLVED_COMMIT:
    raise RuntimeError("Manifest commit does not match the cloned repository")
if manifest.get("parameter_scenarios") != 13 or manifest.get("parameter_artery_rows") != 78:
    raise RuntimeError("Parametric output dimensions are incorrect")
print("Local production run validated.")

## 8. Inspect principal current and parameter tables

In [ ]:
import pandas as pd
from IPython.display import display

current_decisions = pd.read_csv(LOCAL_OUTPUT / "scientific_study" / "assessment" / "primary_current_decisions.csv")
parameter_summary = pd.read_csv(LOCAL_OUTPUT / "parameter_study" / "parameter_scenario_summary.csv")
print("Current-primary hypothesis decisions")
display(current_decisions)
print("Parameter scenario summary")
display(parameter_summary)

## 9. Preview complete multi-panel figures

The displayed PNG files are previews. The Drive archive also contains editable PDF and SVG files and 600 dpi TIFF files for submission.

In [ ]:
from IPython.display import Image, display

for figure_name in (
    "Figure_1_forcing_and_baseline_response.png",
    "Figure_2_current_primary_assessment.png",
    "Figure_3_parametric_robustness.png",
):
    print(figure_name)
    display(Image(filename=str(LOCAL_OUTPUT / "figures" / figure_name), width=1100))

## 10. Copy the complete immutable run to Google Drive

In [ ]:
try:
    for child in LOCAL_OUTPUT.iterdir():
        destination = DRIVE_RUN_ROOT / child.name
        if child.is_dir():
            shutil.copytree(child, destination, dirs_exist_ok=True)
        else:
            shutil.copy2(child, destination)
    shutil.copytree(LOCAL_LOGS, DRIVE_RUN_ROOT / "logs", dirs_exist_ok=True)
except Exception as error:
    record_failure("drive_copy", error)
    raise

## 11. Verify every archived checksum

In [ ]:
checksum_records = json.loads((DRIVE_RUN_ROOT / "SHA256SUMS.json").read_text())
failures = []
for relative_name, record in checksum_records.items():
    path = DRIVE_RUN_ROOT / relative_name
    if not path.is_file():
        failures.append(f"missing: {relative_name}")
        continue
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    if digest != record["sha256"]:
        failures.append(f"checksum mismatch: {relative_name}")
if failures:
    raise RuntimeError("Archive verification failed:\n" + "\n".join(failures))
print(f"Verified {len(checksum_records):,} archived files.")

## 12. Final summary

In [ ]:
manifest = json.loads((DRIVE_RUN_ROOT / "FINAL_MANIFEST.json").read_text())
assessment = json.loads((DRIVE_RUN_ROOT / "scientific_study" / "assessment" / "scientific_assessment.json").read_text())
print("=" * 76)
print("picoNewton_v4 full parametric production run completed")
print("=" * 76)
print("Run ID:", RUN_ID)
print("Repository commit:", RESOLVED_COMMIT)
print("Persistent output:", DRIVE_RUN_ROOT)
print("Status:", manifest["status"])
print("Study outcome:", manifest["study_outcome"])
print("Claims enabled:", manifest["claims_enabled"])
print("Parameter scenarios:", manifest["parameter_scenarios"])
print("Parametric artery rows:", manifest["parameter_artery_rows"])
print("Figure files:", len(manifest["figure_files"]))
print("Checksummed files:", manifest["checksummed_files"])
print("\nCompletion gates:")
for gate, passed in assessment["gates"].items():
    print(f"  {gate}: {passed}")